<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [10]:
import piplite
await piplite.install(['folium'])
await piplite.install(['pandas'])

In [11]:
import folium
import pandas as pd

In [12]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [13]:
# Download and read the `spacex_launch_geo.csv`
from js import fetch
import io

URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
resp = await fetch(URL)
spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
spacex_df=pd.read_csv(spacex_csv_file)

Now, you can take a look at what are the coordinates for each site.


In [14]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [15]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [16]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [13]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# For each launch site, add a Circle object based on its coordinate (Lat, Long) values. In addition, add Launch site name as a popup label


The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


In [21]:
# Create a map centered at NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Create a marker cluster to better visualize many markers in the same area
marker_cluster = MarkerCluster()

# Add markers for each launch
for index, row in spacex_df.iterrows():
    # Define the marker color based on launch success or failure
    if row['class'] == 1:
        marker_color = 'green'  # Changed from 'success' to 'green'
        result = 'Success'
    else:
        marker_color = 'red'    # Changed from 'danger' to 'red'
        result = 'Failure'
    
    # Create a marker with popup information
    marker = folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=f"Launch Site: {row['Launch Site']}<br>Result: {result}",
        icon=folium.Icon(color=marker_color, icon='info-sign')
    )
    
    # Add marker to the cluster
    marker_cluster.add_child(marker)

# Add the marker cluster to the map
site_map.add_child(marker_cluster)

# Display the map
site_map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [19]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [22]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [23]:
# Create a new column 'marker_color' based on the class value
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')

# Display the first few rows to verify our new column
spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [24]:
# Create a map with markers using the new marker_color column
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster()

for index, row in spacex_df.iterrows():
    # Use the marker_color column we just created
    marker = folium.Marker(
        location=[row['Lat'], row['Long']],
        popup=f"Launch Site: {row['Launch Site']}<br>Result: {'Success' if row['class'] == 1 else 'Failure'}",
        icon=folium.Icon(color=row['marker_color'], icon='info-sign')
    )
    marker_cluster.add_child(marker)

site_map.add_child(marker_cluster)
site_map

*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [25]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed, 
# e.g., icon=folium.Icon(color='white', icon_color=row['marker_color']
for index, record in spacex_df.iterrows():
    # TODO: Create and add a Marker cluster to the site map
    # marker = folium.Marker(...)
    marker_cluster.add_child(marker)

site_map

In [26]:
# Create a marker cluster
marker_cluster = MarkerCluster()

# for each row in spacex_df data frame
# create a Marker object with its coordinate
# and customize the Marker's icon property to indicate if this launch was successed or failed
for index, record in spacex_df.iterrows():
    # Create a marker for each launch
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"Launch Site: {record['Launch Site']}<br>Result: {'Success' if record['class'] == 1 else 'Failure'}",
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    # Add the marker to the cluster
    marker_cluster.add_child(marker)

# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [27]:
import numpy as np

# Function to calculate distance between two points on Earth using Haversine formula
def calculate_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points 
    on the Earth specified in decimal degrees of latitude and longitude.
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of earth in kilometers
    return c * r

# Define proximity points of interest (coordinates of cities, landmarks, etc.)
proximities = {
    "Orlando": (28.5383, -81.3792),
    "Tampa": (27.9506, -82.4572),
    "Miami": (25.7617, -80.1918),
    "Jacksonville": (30.3322, -81.6557),
    "Railway": (28.5721, -80.5872),
    "Highway": (28.5638, -80.6277)
}

# Select a launch site to analyze (for example, using the first site in the dataframe)
site = launch_sites_df.iloc[0]  # CCAFS LC-40
site_name = site['Launch Site']
site_lat = site['Lat']
site_long = site['Long']

# Calculate distances to each proximity point
distances = {}
for location_name, (location_lat, location_long) in proximities.items():
    distance = calculate_distance(site_lat, site_long, location_lat, location_long)
    distances[location_name] = distance

# Print the distances
print(f"Distances from {site_name}:")
for location, distance in distances.items():
    print(f"Distance to {location}: {distance:.2f} km")

# Create a map to visualize the distances
proximity_map = folium.Map(location=[site_lat, site_long], zoom_start=8)

# Add a marker for the launch site
folium.Marker(
    location=[site_lat, site_long],
    popup=f"Launch Site: {site_name}",
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(proximity_map)

# Add markers and lines for each proximity location
for location_name, (location_lat, location_long) in proximities.items():
    distance = distances[location_name]
    
    # Add marker for the proximity location
    folium.Marker(
        location=[location_lat, location_long],
        popup=f"{location_name}",
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(proximity_map)
    
    # Add a line connecting the launch site to the proximity location
    folium.PolyLine(
        locations=[[site_lat, site_long], [location_lat, location_long]],
        color='blue',
        weight=2,
        opacity=0.7,
        popup=f"Distance to {location_name}: {distance:.2f} km"
    ).add_to(proximity_map)

# Display the map
proximity_map


Distances from CCAFS LC-40:
Distance to Orlando: 78.36 km
Distance to Tampa: 196.28 km
Distance to Miami: 313.74 km
Distance to Jacksonville: 222.78 km
Distance to Railway: 1.45 km
Distance to Highway: 4.92 km


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [28]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [29]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [30]:
# Choose a launch site (for example, CCAFS LC-40)
launch_site = launch_sites_df.iloc[0]  # Selecting the first launch site
launch_site_lat = launch_site['Lat']
launch_site_lon = launch_site['Long']
launch_site_name = launch_site['Launch Site']

# Create a map centered at the launch site
coastline_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=12)

# Add MousePosition to show coordinates on map hover
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' | ',
    empty_string='NaN',
    lng_first=True,
    num_digits=5,
    prefix='Coordinates:',
    lat_formatter=formatter,
    lng_formatter=formatter
)
coastline_map.add_child(mouse_position)

# Add a marker for the launch site
folium.Marker(
    [launch_site_lat, launch_site_lon],
    popup=f"Launch Site: {launch_site_name}",
    icon=folium.Icon(color='red')
).add_to(coastline_map)

# Display the map so you can find the coastline coordinates
coastline_map

In [31]:
# Use the coordinates you found from the map
# Example coordinates (you should replace with your own findings):
coastline_lat = 28.56367  # Replace with the latitude you identified
coastline_lon = -80.57163  # Replace with the longitude you identified

# Calculate the distance
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# Mark the coastline point on the map
folium.Marker(
    [coastline_lat, coastline_lon],
    popup="Closest Coastline Point",
    icon=folium.Icon(color='blue')
).add_to(coastline_map)

# Draw a line between launch site and coastline
folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    color='green',
    weight=2,
    opacity=1,
    popup=f"Distance to Coastline: {distance_coastline:.2f} km"
).add_to(coastline_map)

# Print the distance
print(f"Distance from {launch_site_name} to closest coastline: {distance_coastline:.2f} km")

# Display the updated map
coastline_map

Distance from CCAFS LC-40 to closest coastline: 0.58 km


In [32]:
# Use the coordinates you identified for the coastline
coastline_lat = 28.56367  # Replace with the actual latitude you found
coastline_lon = -80.57163  # Replace with the actual longitude you found

# Calculate the distance between launch site and coastline
distance = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

# Create a marker at the coastline point
folium.Marker(
    [coastline_lat, coastline_lon],
    popup="Closest Coastline Point",
    icon=folium.Icon(color='blue')
).add_to(coastline_map)

# Create a marker that displays the distance
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance),
        )
    )

# Add the distance marker to the map
coastline_map.add_child(distance_marker)

# Draw a line connecting the launch site to the coastline
folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    color='green',
    weight=2,
    opacity=1
).add_to(coastline_map)

# Display the map
coastline_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [33]:
# Define the coordinates
launch_coordinates = [launch_site_lat, launch_site_lon]
coastline_coordinates = [coastline_lat, coastline_lon]

# Combine the coordinates for the PolyLine
coordinates = [launch_coordinates, coastline_coordinates]

# Create a PolyLine connecting the launch site to the coastline
lines = folium.PolyLine(
    locations=coordinates,
    weight=2,
    color='blue',
    opacity=0.7,
    tooltip=f"Distance: {distance:.2f} km"
)

# Add the PolyLine to the map
site_map.add_child(lines)

# Display the map
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [34]:
# Define coordinates for nearby cities, railways, highways, and other points of interest
landmarks = {
    "City: Orlando": (28.5383, -81.3792),
    "City: Titusville": (28.5919, -80.8075),
    "City: Cape Canaveral": (28.4055, -80.6258),
    "Railway": (28.5721, -80.5872),
    "Highway I-95": (28.5638, -80.8002),
    "Kennedy Space Center": (28.5728, -80.6490)
}

# Select a launch site to analyze
launch_site = launch_sites_df.iloc[0]  # For example, using CCAFS LC-40
launch_site_name = launch_site['Launch Site']
launch_site_lat = launch_site['Lat']
launch_site_long = launch_site['Long']

# Create a map centered at the launch site
landmark_map = folium.Map(location=[launch_site_lat, launch_site_long], zoom_start=9)

# Add a marker for the launch site
folium.Marker(
    [launch_site_lat, launch_site_long],
    popup=f"Launch Site: {launch_site_name}",
    icon=folium.Icon(color='red', icon='rocket', prefix='fa')
).add_to(landmark_map)

# Calculate distances to each landmark and add markers + lines
for landmark_name, (landmark_lat, landmark_long) in landmarks.items():
    # Calculate distance
    distance = calculate_distance(launch_site_lat, launch_site_long, landmark_lat, landmark_long)
    
    # Create a marker for the landmark
    folium.Marker(
        [landmark_lat, landmark_long],
        popup=f"{landmark_name}",
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(landmark_map)
    
    # Create a distance text marker
    distance_marker = folium.Marker(
        [(launch_site_lat + landmark_lat)/2, (launch_site_long + landmark_long)/2],  # Middle of the line
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance:.2f} KM</b></div>'
        )
    )
    landmark_map.add_child(distance_marker)
    
    # Draw a line from the launch site to the landmark
    line = folium.PolyLine(
        locations=[[launch_site_lat, launch_site_long], [landmark_lat, landmark_long]],
        weight=2,
        color='green',
        opacity=0.7,
        tooltip=f"Distance to {landmark_name}: {distance:.2f} km"
    )
    landmark_map.add_child(line)

# Display the map
landmark_map


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
